# Лабораторная 03. Shuffle и `explain("formatted")`

Цель: научиться видеть shuffle в physical plan по оператору `Exchange`.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-03-shuffle-explain').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Narrow operation
`filter/select` обычно не требует shuffle.

In [ ]:
narrow_df = orders.filter(F.col('status') == 'paid').select('order_id', 'customer_id')
narrow_df.explain('formatted')

Вопрос: есть ли `Exchange`? Почему?

Ответ: ...

## groupBy
`groupBy` должен собрать одинаковые ключи вместе.

In [ ]:
by_customer = orders.groupBy('customer_id').count()
by_customer.explain('formatted')
by_customer.count()

Вопросы:

- Где появился `Exchange`?
- Почему `Exchange` связан с shuffle?
- Сколько shuffle partitions задано?

Ответ: ...

## join
Отключим broadcast, чтобы увидеть shuffle join.

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
joined = orders.join(customers, 'customer_id')
joined.explain('formatted')
joined.count()

Вопросы:

- Какой join выбрал Spark?
- Сколько `Exchange` в плане?
- Почему join часто требует shuffle?

Ответ: ...